In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip


In [4]:
import os
import sys
import logging
import warnings

import pandas as pd
import numpy as np

import datasets
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, DataCollatorWithPadding
from transformers import Trainer, TrainingArguments
from sklearn.model_selection import train_test_split

# -------------------------- Kaggle 路径配置 --------------------------
PATH_LABELED_TRAIN = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
PATH_TEST = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"

# 创建输出文件夹
os.makedirs("./results", exist_ok=True)
os.makedirs("./result", exist_ok=True)

warnings.filterwarnings("ignore")

# 日志配置
program = os.path.basename(sys.argv[0])
logger = logging.getLogger(program)
logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
logging.root.setLevel(level=logging.INFO)
logger.info(f"running {''.join(sys.argv)}")


train = pd.read_csv(PATH_LABELED_TRAIN, header=0, delimiter="\t", quoting=3)
test = pd.read_csv(PATH_TEST, header=0, delimiter="\t", quoting=3)

logger.info(f"train shape: {train.shape}, test shape: {test.shape}")

train_df, val_df = train_test_split(train, test_size=0.2, random_state=42)

train_dict = {'label': train_df["sentiment"], 'text': train_df['review']}
val_dict = {'label': val_df["sentiment"], 'text': val_df['review']}
test_dict = {"text": test['review']}

train_dataset = datasets.Dataset.from_dict(train_dict)
val_dataset = datasets.Dataset.from_dict(val_dict)
test_dataset = datasets.Dataset.from_dict(test_dict)

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True)

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')

# 手动计算accuracy，不依赖evaluate库
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    correct = np.sum(predictions == labels)
    total = len(labels)
    acc = correct / total
    return {"accuracy": acc}


training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=12,
    per_device_eval_batch_size=24,
    warmup_steps=500,
    weight_decay=0.01,
    # 删掉已废弃 logging_dir
    logging_steps=100,
    save_strategy="no",
    eval_strategy="epoch",
    fp16=True,
    report_to="none"
)

# ✅ 删除 Trainer 的 tokenizer=tokenizer 参数，新版本不支持
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

prediction_outputs = trainer.predict(tokenized_test)
test_pred = np.argmax(prediction_outputs.predictions, axis=-1).flatten()

result_output = pd.DataFrame(data={"id": test["id"], "sentiment": test_pred})
result_output.to_csv("./result/distilbert_trainer.csv", index=False, quoting=3)
logger.info('result saved!')

trainer.save_model("./results/distilbert-imdb-final")
tokenizer.save_pretrained("./results/distilbert-imdb-final")

print("✅完成！提交文件路径： ./result/distilbert_trainer.csv")
print(result_output.head())


2026-08-28 07:26:54,601: INFO: running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py-f/root/.local/share/jupyter/runtime/kernel-57f21b71-90a5-48c5-9e81-5e14fb0e42ac.json
2026-08-28 07:26:55,951: INFO: train shape: (25000, 3), test shape: (25000, 2)
2026-08-28 07:26:56,643: INFO: HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-28 07:26:56,644: WARNING: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-08-28 07:26:56,756: INFO: HTTP Request: GET https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

2026-08-28 07:26:56,834: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-08-28 07:26:56,899: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-28 07:26:56,959: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-08-28 07:26:57,023: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-08-28 07:26:57,094: INFO: HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/vocab.txt "HTTP/1.1 200 OK"
2026-08-28 07:26:57,159: INFO: HTTP Request: GET https://huggingface.co/distilbert-base-uncased

vocab.txt: 0.00B [00:00, ?B/s]

2026-08-28 07:26:57,261: INFO: HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer.json "HTTP/1.1 200 OK"
2026-08-28 07:26:57,331: INFO: HTTP Request: GET https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer.json "HTTP/1.1 200 OK"


...

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.477972,0.436274,0.910800
2,0.318411,0.525862,0.919000
3,0.118188,0.636225,0.925600


2026-08-28 07:59:35,464: INFO: result saved!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅完成！提交文件路径： ./result/distilbert_trainer.csv
           id  sentiment
0  "12311_10"          1
1    "8348_2"          0
2    "5828_4"          0
3    "7186_2"          0
4   "12128_7"          1


In [3]:
import torch

# 检测CUDA/GPU
print(f"torch版本: {torch.__version__}")
print(f"CUDA是否可用: {torch.cuda.is_available()}")

# 自动选择设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用设备: {device}")

if torch.cuda.is_available():
    print(f"GPU型号: {torch.cuda.get_device_name(0)}")
    print(f"GPU数量: {torch.cuda.device_count()}")
else:
    print("⚠️ 警告：没有检测到GPU！请在右侧设置 -> Accelerator 选择GPU T4，然后重启notebook")


torch版本: 2.10.0+cu128
CUDA是否可用: True
当前使用设备: cuda
GPU型号: Tesla T4
GPU数量: 2


In [8]:
import os
import sys
import logging
import warnings
import time
import random

import pandas as pd
import numpy as np
import torch
import datasets
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, DataCollatorWithPadding
from transformers import Trainer, TrainingArguments
from sklearn.model_selection import train_test_split

# ====================== 1. 环境与GPU校验 ======================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch version: {torch.__version__}")
print(f"Cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: No GPU detected! Go to Settings -> Accelerator select GPU T4 and restart kernel")

use_fp16 = torch.cuda.is_available()

os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore")

# -------------------------- Kaggle 路径配置 --------------------------
PATH_LABELED_TRAIN = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
PATH_TEST = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"

os.makedirs("./results", exist_ok=True)
os.makedirs("./result", exist_ok=True)

program = os.path.basename(sys.argv[0])
logger = logging.getLogger(program)
logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
logging.root.setLevel(level=logging.INFO)
logger.info(f"running {''.join(sys.argv)}")

# ====================== 2. 加载数据 ======================
train = pd.read_csv(PATH_LABELED_TRAIN, header=0, delimiter="\t", quoting=3)
test = pd.read_csv(PATH_TEST, header=0, delimiter="\t", quoting=3)

logger.info(f"train shape: {train.shape}, test shape: {test.shape}")

train_df, val_df = train_test_split(train, test_size=0.2, random_state=SEED)

train_dict = {'label': train_df["sentiment"], 'text': train_df['review']}
val_dict = {'label': val_df["sentiment"], 'text': val_df['review']}
# 关键修复：test_dict保留id字段，保证顺序
test_dict = {"id": test["id"], "text": test['review']}

train_dataset = datasets.Dataset.from_dict(train_dict)
val_dataset = datasets.Dataset.from_dict(val_dict)
test_dataset = datasets.Dataset.from_dict(test_dict)

logger.info(f"train samples:{len(train_dataset)}, val samples:{len(val_dataset)}, test samples:{len(test_dataset)}")

# ====================== 3. Tokenize ======================
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512)

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')

# ====================== 4. 评价指标 ======================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    correct = np.sum(predictions == labels)
    total = len(labels)
    acc = correct / total
    return {"accuracy": round(acc, 4)}

# ====================== 5. Trainer 参数 ======================
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=12,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=100,
    save_strategy="no",
    eval_strategy="epoch",
    fp16=use_fp16,
    report_to="none",
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# ====================== 6. 训练 + 计时 ======================
start_time = time.time()
trainer.train()
train_cost = time.time() - start_time
logger.info(f"Training finished, cost {train_cost:.2f}s")

if torch.cuda.is_available():
    print(f"GPU max memory allocated: {torch.cuda.max_memory_allocated() / 1024 / 1024:.1f} MB")

# ====================== 7. 预测 & 输出竞赛提交文件【修复id顺序】 ======================
pred_start = time.time()
prediction_outputs = trainer.predict(tokenized_test)
test_pred = np.argmax(prediction_outputs.predictions, axis=-1).flatten()
logger.info(f"Inference cost {time.time()-pred_start:.2f}s")

# 从tokenized_test拿id，和预测结果严格一一对应
result_output = pd.DataFrame({
    "id": tokenized_test["id"],
    "sentiment": test_pred
})

result_output.to_csv("./result/distilbert_trainer.csv", index=False)
logger.info('Result csv saved!')

trainer.save_model("./results/distilbert-imdb-final")
tokenizer.save_pretrained("./results/distilbert-imdb-final")

print("\n✅Done! Submit file path: ./result/distilbert_trainer.csv")
print(result_output.head())
print(f"输出总行数 {len(result_output)}")


2026-08-28 09:23:45,266: INFO: running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py-f/root/.local/share/jupyter/runtime/kernel-57f21b71-90a5-48c5-9e81-5e14fb0e42ac.json


Torch version: 2.10.0+cu128
Cuda available: True
GPU: Tesla T4


2026-08-28 09:23:46,367: INFO: train shape: (25000, 3), test shape: (25000, 2)
2026-08-28 09:23:46,697: INFO: train samples:20000, val samples:5000, test samples:25000
2026-08-28 09:23:46,837: INFO: HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-28 09:23:46,901: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-08-28 09:23:46,972: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-28 09:23:47,033: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-08-28 09:23:47,096: INFO: HTTP Request: GET https://huggingface.co/api/models/distil

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

...

2026-08-28 09:53:09,594: INFO: Training finished, cost 1735.41s


GPU max memory allocated: 3794.7 MB


2026-08-28 09:56:39,155: INFO: Inference cost 209.56s
2026-08-28 09:56:39,456: INFO: Result csv saved!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅Done! Submit file path: ./result/distilbert_trainer.csv
           id  sentiment
0  "12311_10"          1
1    "8348_2"          0
2    "5828_4"          0
3    "7186_2"          0
4   "12128_7"          1
输出总行数 25000


In [1]:
import os
import sys
import logging
import warnings
import zipfile

import pandas as pd
import numpy as np
import torch

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, DataCollatorWithPadding
from transformers import Trainer, TrainingArguments
from sklearn.model_selection import train_test_split

PATH_LABELED_TRAIN = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
PATH_TEST = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
PATH_SAMPLE = "/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv"

os.makedirs("./results", exist_ok=True)
os.makedirs("./result", exist_ok=True)
warnings.filterwarnings("ignore")

# 解压读取zip内部tsv，规避pandas直接读zip丢行bug
with zipfile.ZipFile(PATH_LABELED_TRAIN) as zf:
    with zf.open("labeledTrainData.tsv") as f:
        train = pd.read_csv(f, sep="\t", quoting=3)

with zipfile.ZipFile(PATH_TEST) as zf:
    with zf.open("testData.tsv") as f:
        test = pd.read_csv(f, sep="\t", quoting=3)

sample = pd.read_csv(PATH_SAMPLE)
print(f"test行数:{len(test)}, sample行数:{len(sample)}")
assert len(test)==25000, f"test数据集读取错误！实际读取只有{len(test)}行"

train_df, val_df = train_test_split(train, test_size=0.2, random_state=42)

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

#训练部分用dataset
import datasets
train_dataset = datasets.Dataset.from_dict({"text":train_df['review'],"label":train_df['sentiment']})
val_dataset = datasets.Dataset.from_dict({"text":val_df['review'],"label":val_df['sentiment']})

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": np.mean(predictions==labels)}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=12,
    per_device_eval_batch_size=24,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=100,
    save_strategy="no",
    eval_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# 手动batch推理，严格遵循test表顺序
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval().to(device)
all_preds = []
batch_size =24
reviews = test["review"].tolist()

with torch.no_grad():
    for i in range(0, len(reviews), batch_size):
        batch_text = reviews[i:i+batch_size]
        batch_enc = tokenizer(batch_text, truncation=True, padding=True, return_tensors="pt").to(device)
        logits = model(**batch_enc).logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())

result_output = pd.DataFrame({
    "id": test["id"],
    "sentiment": all_preds
})

# 强制标准输出参数
result_output.to_csv(
    "./result/distilbert_trainer.csv",
    index=False,
    sep=",",
    encoding="utf-8",
    lineterminator="\n"
)

# 自检！！！运行完必须看输出
my_out = pd.read_csv("./result/distilbert_trainer.csv")
print(f"\n输出文件行数 {len(my_out)}")
print("前5个id", my_out['id'].head().tolist())
assert len(my_out)==25000, "输出结果行数不对！！"
assert set(my_out["id"]) == set(sample["id"]), "id集合和官方样例不一致！"

print("✅全部自检通过，可以提交！")



test行数:25000, sample行数:25000


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.492143,0.458425,0.908400
2,0.321780,0.523674,0.918200
3,0.115233,0.653210,0.929200



输出文件行数 25000
前5个id ['"12311_10"', '"8348_2"', '"5828_4"', '"7186_2"', '"12128_7"']


AssertionError: id集合和官方样例不一致！